### Kontrollera pris vs tid på dygn och veckodag

Jag har byggt klart min applikation, men jag tycker att prisprediktionen är rätt dålig.    
Det borde vara större skillnad på priset beroende av vilken tid på dygnet eller veckodag som kunden vill åka.   
Frågan är om prediktionen beror på datasetet som sådant, eller om jag gjort fel när jag antingen rensat datasetet, tränat modellen eller skapat endpointen.   

Jag går därför tillbaka till datasetet och undersöker korrelation mellan pris och vilken tid på dygnet eller veckodag som resan görs.


In [26]:
import pandas as pd
import matplotlib.pyplot as plt
from taxipred.utils.constants import DATA_PATH

In [ ]:
#Läser in datasetet på nytt och skapar en ny dataframe
df = pd.read_csv(DATA_PATH / "taxi_trip_pricing.csv")

df.head()

,Trip_Distance_km,Time_of_Day,Day_of_Week,Passenger_Count,Traffic_Conditions,Weather,Base_Fare,Per_Km_Rate,Per_Minute_Rate,Trip_Duration_Minutes,Trip_Price
0,19.35,Morning,Weekday,3.0,Low,Clear,3.56,0.80,0.32,53.82,36.2624
1,47.59,Afternoon,Weekday,1.0,High,Clear,NaN,0.62,0.43,40.57,NaN
2,36.87,Evening,Weekend,1.0,High,Clear,2.70,1.21,0.15,37.27,52.9032
3,30.33,Evening,Weekday,4.0,Low,NaN,3.48,0.51,0.15,116.81,36.4698
4,NaN,Evening,Weekday,3.0,High,Clear,2.93,0.63,0.32,22.64,15.6180


In [71]:
#En df där jag droppar alla rader med nullvärden
df_drop = df.dropna()
df_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 562 entries, 0 to 998
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Trip_Distance_km       562 non-null    float64
 1   Time_of_Day            562 non-null    object 
 2   Day_of_Week            562 non-null    object 
 3   Passenger_Count        562 non-null    float64
 4   Traffic_Conditions     562 non-null    object 
 5   Weather                562 non-null    object 
 6   Base_Fare              562 non-null    float64
 7   Per_Km_Rate            562 non-null    float64
 8   Per_Minute_Rate        562 non-null    float64
 9   Trip_Duration_Minutes  562 non-null    float64
 10  Trip_Price             562 non-null    float64
 11  DayTime                562 non-null    object 
dtypes: float64(7), object(5)
memory usage: 57.1+ KB


In [72]:
#En df där jag ersätter alla numeriska nullvärden med medianvärde. 
df_fill = df.fillna(df.median(numeric_only=True))
df_fill.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Trip_Distance_km       1000 non-null   float64
 1   Time_of_Day            950 non-null    object 
 2   Day_of_Week            950 non-null    object 
 3   Passenger_Count        1000 non-null   float64
 4   Traffic_Conditions     950 non-null    object 
 5   Weather                950 non-null    object 
 6   Base_Fare              1000 non-null   float64
 7   Per_Km_Rate            1000 non-null   float64
 8   Per_Minute_Rate        1000 non-null   float64
 9   Trip_Duration_Minutes  1000 non-null   float64
 10  Trip_Price             1000 non-null   float64
 11  DayTime                901 non-null    object 
dtypes: float64(7), object(5)
memory usage: 93.9+ KB


In [75]:
#Fyller kategoriska kolumner med Unknown
df_fill["Time_of_Day"] = df_fill["Time_of_Day"].fillna("Unknown")
df_fill["Day_of_Week"] = df_fill["Day_of_Week"].fillna("Unknown")
df_fill["Traffic_Conditions"] = df_fill["Traffic_Conditions"].fillna("Unknown")
df_fill["Weather"] = df_fill["Weather"].fillna("Unknown")

df_fill.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Trip_Distance_km       1000 non-null   float64
 1   Time_of_Day            1000 non-null   object 
 2   Day_of_Week            1000 non-null   object 
 3   Passenger_Count        1000 non-null   float64
 4   Traffic_Conditions     1000 non-null   object 
 5   Weather                1000 non-null   object 
 6   Base_Fare              1000 non-null   float64
 7   Per_Km_Rate            1000 non-null   float64
 8   Per_Minute_Rate        1000 non-null   float64
 9   Trip_Duration_Minutes  1000 non-null   float64
 10  Trip_Price             1000 non-null   float64
 11  DayTime                901 non-null    object 
dtypes: float64(7), object(5)
memory usage: 93.9+ KB


In [76]:
df_drop.describe().T

,count,mean,std,min,25%,50%,75%,max
Trip_Distance_km,562.0,27.772941,21.153175,1.2700,13.135000,26.42000,38.827500,146.067047
Passenger_Count,562.0,2.533808,1.108915,1.0000,2.000000,3.00000,4.000000,4.000000
Base_Fare,562.0,3.509893,0.871082,2.0100,2.722500,3.54500,4.260000,5.000000
Per_Km_Rate,562.0,1.219858,0.430351,0.5000,0.840000,1.20000,1.580000,2.000000
Per_Minute_Rate,562.0,0.288221,0.114834,0.1000,0.190000,0.28000,0.387500,0.500000
Trip_Duration_Minutes,562.0,61.825089,32.128436,5.0100,36.530000,61.21000,88.435000,119.840000
Trip_Price,562.0,57.663525,43.958741,6.1269,33.583875,50.15785,69.146575,332.043689


In [77]:
df_fill.describe().T

,count,mean,std,min,25%,50%,75%,max
Trip_Distance_km,1000.0,27.00852,19.402661,1.2300,13.10750,25.8300,37.78250,146.067047
Passenger_Count,1000.0,2.45300,1.079331,1.0000,2.00000,2.0000,3.00000,4.000000
Base_Fare,1000.0,3.50384,0.848115,2.0100,2.77000,3.5200,4.20250,5.000000
Per_Km_Rate,1000.0,1.23265,0.418932,0.5000,0.87000,1.2200,1.58000,2.000000
Per_Minute_Rate,1000.0,0.29277,0.112664,0.1000,0.19750,0.2900,0.38250,0.500000
Trip_Duration_Minutes,1000.0,62.10521,31.339464,5.0100,37.10750,61.8600,87.77500,119.840000
Trip_Price,1000.0,56.54156,39.492129,6.1269,34.57885,50.0745,67.47665,332.043689


Har nu tre olika dataframes:   
- df = rådata
- df_fill = df där jag fyllt alla NaN
- df_drop = df där jag tagit bort alla NaN

In [80]:
median_df = df.groupby("Day_of_Week")["Trip_Price"].median()
median_fill = df_fill.groupby("Day_of_Week")["Trip_Price"].median()
median_drop = df_drop.groupby("Day_of_Week")["Trip_Price"].median()


In [82]:
print(f"Median för Trip_Price (rådata): {median_df}")
print("")
print(f"Median för Trip_Price (df_fill): {median_fill}")
print("")
print(f"Median för Trip_Price (df_drop): {median_drop}")

Median för Trip_Price (rådata): Day_of_Week
Weekday    51.0126
Weekend    47.4882
Name: Trip_Price, dtype: float64

Median för Trip_Price (df_fill): Day_of_Week
Unknown    50.0745
Weekday    50.0745
Weekend    49.5082
Name: Trip_Price, dtype: float64

Median för Trip_Price (df_drop): Day_of_Week
Weekday    52.0972
Weekend    46.1779
Name: Trip_Price, dtype: float64


Utifrån ovanstående ser jag ganska tydligt att det - i det här fallet - inte har varit gynnsamt att enbart fylla nullvärden med median-värden.    
Även om jag har mer data att utgå från, så är priskillnaden mellan Weekday och Weekend i princip borta. 
Frågan är om det samma även gäller tidpunkt på dygnet?


In [101]:
df["Time_of_Day"].unique()

array(['Morning', 'Afternoon', 'Evening', 'Night', nan], dtype=object)

In [94]:
tod_df = df.groupby("Time_of_Day")["Trip_Price"].median()
tod_df_fill = df_fill.groupby("Time_of_Day")["Trip_Price"].median()
tod_df_drop = df_drop.groupby("Time_of_Day")["Trip_Price"].median()



In [95]:
print(f"Rådata: {tod_df.describe()}")
print()
print(f"Df_fill: {tod_df_fill.describe()}")
print()
print(f"Df_drop: {tod_df_drop.describe()}")

Rådata: count     4.000000
mean     50.370725
std       1.299716
min      48.449600
25%      50.160800
50%      50.880600
75%      51.090525
max      51.272100
Name: Trip_Price, dtype: float64

Df_fill: count     5.000000
mean     49.986180
std       0.129732
min      49.787300
25%      49.920100
50%      50.074500
75%      50.074500
max      50.074500
Name: Trip_Price, dtype: float64

Df_drop: count     4.000000
mean     51.136562
std       2.177411
min      49.819650
25%      49.845712
50%      50.179100
75%      51.469950
max      54.368400
Name: Trip_Price, dtype: float64


Här är det svårare att avgöra om fill eller drop är bäst att välja.   
Med fill blir det väldigt liten avvikelse.   
Med drop blir avvikelsen högre än i rådatan. 

##### Tips från chatGPT - använda scipy.stats för att se om det finns ett statistiskt signifikant sammanhang eller inte. 
Funktionen **ttest_ind** gör att jag kan jämföra två olika oberoende grupper av resor (ex. morgon och kväll) och se om medelvärdet skiljer sig signifikant mellan grupperna.  
         
- **t-statistic** - visar skillnaden mellan gruppernas medelvärden i förhållande till spridningen. Ju längre från 0, desto större faktisk skillnad. 
    - ≈ 0	Grupperna är nästan identiska.
    - ±1–2	Små skillnader (ofta ej signifikanta).
    - ±2–3	Måttliga skillnader (ofta signifikanta vid p < 0.05).
    - mer än ±3	Stora skillnader mellan grupper.   

- **p-value** - visar sannolikheten för att värdet är en slump
    - mer än 0.05 = Mer än 5 % sannolikhet att skillnanden kan ha uppstått av en slump (svag signifikans)
    - < 0.05 = Mindre än 5 % sannolikhet att skillnanden kan ha uppstått av en slump (högre signifikans)   

- **df** = hur mycket data som testat baseras på


In [ ]:
import scipy.stats as stats

result_drop = stats.ttest_ind(
    df_drop[df_drop["Time_of_Day"] == "Morning"]["Trip_Price"],
    df_drop[df_drop["Time_of_Day"] == "Night"]["Trip_Price"]
)
print(f"T-test mellan Morning och Night:")
print(f"t-statistic = {result_drop.statistic:.3f}")
print(f"p-value     = {result_drop.pvalue:.4f}")
print(f"df          = {result_drop.df:.0f}")


T-test mellan Morning och Night:
t-statistic = -0.301
p-value     = 0.7638
df          = 216


In [114]:
import scipy.stats as stats

result_fill = stats.ttest_ind(
    df_fill[df_fill["Time_of_Day"] == "Evening"]["Trip_Price"],
    df_fill[df_fill["Time_of_Day"] == "Night"]["Trip_Price"]
)
print(f"T-test mellan Morning och Night:")
print(f"t-statistic = {result_fill.statistic:.3f}")
print(f"p-value     = {result_fill.pvalue:.4f}")
print(f"df          = {result_fill.df:.0f}")


T-test mellan Morning och Night:
t-statistic = 0.038
p-value     = 0.9693
df          = 294


In [115]:
weekday_prices = df_drop[df_drop["Day_of_Week"] == "Weekday"]["Trip_Price"]
weekend_prices = df_drop[df_drop["Day_of_Week"] == "Weekend"]["Trip_Price"]

result_wd = stats.ttest_ind(weekday_prices, weekend_prices)

print(f"T-test mellan Morning och Night:")
print(f"t-statistic = {result_wd.statistic:.3f}")
print(f"p-value     = {result_wd.pvalue:.4f}")
print(f"df          = {result_wd.df:.0f}")

T-test mellan Morning och Night:
t-statistic = 1.584
p-value     = 0.1137
df          = 560


### Slutsats
Resultaten från testerna visar att varken tid på dygnet eller veckodag har någon större statistiskt signifikans avseende priset.   
Skillnaderna mellan grupperna (exempelvis weekday/weekend och morning/afternoon) är små, och kan förklaras av slumpvariation snarare än riktiga mönster.   
Det tyder på att den begränsade variationen i basdatan gör att modellen inte kan lära sig ett tydligt samband mellan pris och dessa variabler.   
Jag kommer därför inte ändra den data som min modell tidigare har tränats på. 